# Fine-tune WangchanBERTa สำหรับ Auto Engagement API

Fine-tune [WangchanBERTa](https://huggingface.co/airesearch/wangchanberta-base-att-spm-uncased) เป็นตัวจำแนก
**sentiment** และ **category** แล้ว export เป็น ONNX ให้ API ใช้งานบน CPU

**ก่อนเริ่ม:** Runtime → Change runtime type → **T4 GPU**
รัน cell ทีละอัน (กด ▶) ตามลำดับ STEP 0 → 5 ใช้เวลารวมประมาณ 25–30 นาที

ถ้าเน็ตหลุดจน Colab ตัดการเชื่อมต่อ ต้องเริ่มใหม่ตั้งแต่ STEP 1

In [ ]:
# STEP 0: เช็คว่าได้ GPU (ต้องเห็นคำว่า Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# STEP 1: ดึงโค้ดและติดตั้ง library (~2 นาที) — ข้อความ ERROR เรื่อง diffusers/gradio ไม่กระทบ
!git clone -q https://github.com/67130043-tech/auto-engagement-api-new-version.git
%cd auto-engagement-api-new-version
!pip install -q -r requirements.txt -r requirements-train.txt

In [ ]:
# STEP 2: Fine-tune sentiment (Wisesight + Wongnai) ~8 นาที
!python 11_finetune_wangchanberta.py --task sentiment --epochs 3

In [ ]:
# STEP 3: Fine-tune category (10 หมวด) ~6 นาที
!python 11_finetune_wangchanberta.py --task category --epochs 4

ท้าย STEP 2 และ 3 จะมีบรรทัด **`ONNX (ตัวที่ API ใช้จริง)`** — ค่าเริ่มต้นเป็น fp32 (~420MB ต่อโมเดล) ไม่เสียความแม่นยำ
(ทดสอบแล้วว่าการบีบอัด int8 ทำให้ accuracy ตก 5–7 จุด จึงไม่ใช้)

In [ ]:
# (ใช้เมื่อเทรนด้วยโค้ดเวอร์ชันเก่าที่บีบอัด int8 ไว้) export ใหม่แบบ fp32 จากโมเดลที่บันทึกไว้ ไม่ต้องเทรนใหม่
!git pull -q
!python 11_finetune_wangchanberta.py --task sentiment --from-pytorch models/wangchanberta_sentiment_pytorch --quant none
!python 11_finetune_wangchanberta.py --task category --from-pytorch models/wangchanberta_category_pytorch --quant none

In [ ]:
# STEP 4: เทียบผลทุกระบบบน test set เดียวกัน (~3 นาที)
!python 12_compare_models.py
!cut -d, -f1-7 outputs/model_comparison.csv

## STEP 5: อัปโหลดโมเดลขึ้น HuggingFace Hub
ต้องมีบัญชี https://huggingface.co และ Access Token แบบ **Write** (Settings → Access Tokens)
1. รัน cell ด้านล่าง แล้ววาง token ในช่องที่ขึ้นมา
2. แก้ `HF_USER` เป็นชื่อบัญชีของคุณ แล้วรัน cell ถัดไป

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HF_USER = "your-hf-username"   # <- แก้เป็นชื่อบัญชี HuggingFace ของคุณ
from huggingface_hub import HfApi
api = HfApi()
for task in ["sentiment", "category"]:
    repo = f"{HF_USER}/wangchanberta-autoengage-{task}"
    api.create_repo(repo, exist_ok=True)
    api.upload_folder(folder_path=f"models/wangchanberta_{task}", repo_id=repo)
    print("อัปโหลดแล้ว:", f"https://huggingface.co/{repo}")
print("\nตั้งค่าบน Render:")
print(f"WANGCHAN_SENTIMENT_REPO = {HF_USER}/wangchanberta-autoengage-sentiment")
print(f"WANGCHAN_CATEGORY_REPO  = {HF_USER}/wangchanberta-autoengage-category")

In [ ]:
# (ไม่บังคับ) ดาวน์โหลดผลลัพธ์ลงเครื่อง — โมเดล + ตารางเทียบผล
!zip -qr wangchanberta_models.zip models/wangchanberta_sentiment models/wangchanberta_category outputs/model_comparison.csv
from google.colab import files
files.download("wangchanberta_models.zip")

## นำไปใช้กับ API
ตั้ง Environment Variable บน Render ตามที่ STEP 5 พิมพ์ออกมา (ถ้า repo เป็น private ให้ตั้ง `HF_TOKEN` ด้วย)
แล้ว deploy ใหม่ เปิด `/` ของ API ถ้าเห็น `"model": "wangchanberta"` แปลว่าใช้งานได้แล้ว